# Carga del archivo e Importación de Librerías

In [1]:

import numpy as np 
import pandas as pd 
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest # Prueba de z de proporcines 
from statsmodels.stats.multitest import multipletests #para corrección bonferroni

df = pd.read_excel("diabetes.xlsx")
pd.set_option("display.max_rows", None)  # Mostrar todas las filas
pd.set_option("display.max_columns", None)  # Mostrar todas las columnas

# Punto 1

¿El  valor  promedio  de  colesterol  total  de  la  muestra,  correspondiente  a 
pacientes masculinos es menor 200mg/dL Y en las pacientes femeninas, es 
menor 200mg/dL?

Queremos comparar si la media de una muestra es diferente a un valor medio de referencia, tanto para hombres como para mujeres. Por lo tanto la _**prueba t para una sola muestra**_ nos puede servir.

- Variable: s1 $\rightarrow$ total serum cholesterol


- Hipótesis:
    - $H_0$: El valor  promedio  de  colesterol  total  de  la  muestra es igual o mayor a 200 mg/dL para pacientes másculinos y femeninos.
    - $H_1$: El valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

La prueba t, supone que los datos tienen una distribución normal. Para probar la normalidad de los datos, hacemos una prueba de normalidad.

In [2]:
numero_masc = len(df[df['SEX'] == 2]['S1'])

print(f"Total Masculino {numero_masc}")

numero_fem = len(df[df['SEX'] == 1]['S1'])

print(f"Total Femenino {numero_fem}")

Total Masculino 207
Total Femenino 235


Como el total de datos es mayor a 50 en ambos casos, usamos la prueba de Kolmogorov-Smirnoff.

- Definición de Hipótesis para Kolmogorov-Smirnoff
    - $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
    - $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [6]:
alpha = 0.05

datos_hombres_s1 = df[df['SEX'] == 2]['S1']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_hombres_s1), np.std(datos_hombres_s1)))

print("Para hombres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos son ANORMALES.")

#------------------------------------------------------------

datos_mujeres_s1 = df[df['SEX'] == 1]['S1']
ks_stat, ks_p = stats.kstest(datos_hombres_s1, 'norm', args=(np.mean(datos_mujeres_s1), np.std(datos_mujeres_s1)))

print("Para mujeres:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos son ANORMALES.")

Para hombres:
El p-valor (0.2396770979328262) es mayor a 0.05. Asumimos que los datos de S1 en hombres son NORMALES.
Para mujeres:
El p-valor (0.761722343708202) es mayor a 0.05. Asumimos que los datos de S1 en mujeres son NORMALES.


Como en ambos casos, la distribución es normal, podemos efectuar la prueba t

In [ ]:
mu = 200 # (mg/dL) El valor de referencia contra el cual comparamos 
alpha = 0.05

# Para Hombres
datos_hombres_s1 = df[df['SEX'] == 2]['S1']
t_stat, V_pMasc = stats.ttest_1samp(datos_hombres_s1, mu, alternative='less')

# Para Mujeres
datos_mujeres_s1 = df[df['SEX'] == 1]['S1'] # Datos de colesterol total para hombres
t_stat, V_pFem = stats.ttest_1samp(datos_mujeres_s1, mu, alternative='less')

if V_pMasc < alpha:
    print("Para Hombres H0 es Falso")
else:
    print("Para Hombres H0 es Verdadero")

if V_pFem < alpha:
    print("Para Mujeres H0 es Falso")
else:
    print("Para Mujeres H0 es Verdadero")


Para Hombres H0 es Falso
Para Mujeres H0 es Falso


Por lo tanto, tanto para hombres y mujeres, $H_1$ es verdadero; lo que significa que el valor  promedio  de  colesterol  total  de  la  muestra es menor a 200mg/dL para pacientes másculinos y femeninos.

# Punto 2

¿Hay diferencias entre el nivel de colesterol ldl ["S2"] entre pacientes con menos 
de 40 años y más de 40 años de edad?

Compararemos 2 grupos...
 
- Grupo 1: Pacientes con menos de 40 años.
- Grupo 2: Pacientes con más de 40 años.

Es decir, Evalúa si la diferencia de la media de dos muestras es significativamente diferentes de cero. Por lo tanto usaremos una _**Prueba T para dos muestras**_.

Determinamos la cantidad de pacientes mayores y menores a 40 años.

- Hipótesis:
    - $H_0$: La media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son iguales.
    - $H_1$: La media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son diferentes.

La _**Prueba T para dos muestras**_, tiene el supuesto de normalidad en las variables, homocedasticidad e independencia. De esta manera...

- Se asume el supuesto de independencia ya que los datos corresponden a individuos distintos y la medición de uno no influye en la del otro.

- Se asume normalidad en los datos. Así que hacemos una prueba de normalidad...

In [8]:
pacientes_mayores_40 = len(df[df['AGE'] > 40])
print(f"Total de pacientes con edad mayor a 40: {pacientes_mayores_40}")

pacientes_menores_40 = len(df[df['AGE'] < 40])
print(f"Total de pacientes con edad menor a 40: {pacientes_menores_40}")

Total de pacientes con edad mayor a 40: 320
Total de pacientes con edad menor a 40: 117


Como el total de datos es mayor a 50 en ambos casos, usamos la prueba de Kolmogorov-Smirnoff.

- Definición de Hipótesis para Kolmogorov-Smirnoff
    - $H_0$(Hipótesis Nula): Los datos sí siguen una distribución normal.
    - $H_1$(Hipótesis Alterna): Los datos no siguen una distribución normal.

In [10]:
alpha = 0.05

pacientes_mayores_40 = df[df['AGE'] > 40]['S2']
ks_stat, ks_p = stats.kstest(pacientes_mayores_40, 'norm', args=(np.mean(pacientes_mayores_40), np.std(pacientes_mayores_40)))

print("Para Mayores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S2 son ANORMALES.")

# ------------------------------------

pacientes_menores_40 = df[df['AGE'] < 40]['S2']
ks_stat, ks_p = stats.kstest(pacientes_menores_40, 'norm', args=(np.mean(pacientes_menores_40), np.std(pacientes_menores_40)))

print("Para Menores de 40 años:")
if ks_p > alpha:
    print(f"El p-valor ({ks_p}) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.")
else:
    print(f"El p-valor ({ks_p}) es menor a 0.05. Asumimos que los datos de S2 son ANORMALES.")

Para Mayores de 40 años:
El p-valor (0.4272314401788271) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.
Para Menores de 40 años:
El p-valor (0.23024405303211082) es mayor a 0.05. Asumimos que los datos de S2 en hombres son NORMALES.


Como ambos casos tienen una distribución normal, podemos efectuar una _**Prueba T para dos muestras**_.

In [13]:
datos1 = df[df['AGE'] > 40]['S2']
datos2 = df[df['AGE'] < 40]['S2']
t_stat, p_value = stats.ttest_ind(datos1, datos2)

if p_value < alpha:
    print("Para Hombres H0 es Falso")
else:
    print("Para Hombres H0 es Verdadero")

Para Hombres H0 es Falso


Por lo tanto, $H_1$ es verdadero; lo que indica que la media del nivel de colesterol ldl entre pacientes con menos de 40 años y más de 40 años de edad son diferentes.

# Punto 3

¿Hay  diferencia  en  la  medida  de  progresión  de  la  enfermedad  entre 
hombre y mujeres? ¿y entre pacientes con menos de 40 años y más de 40 
años de edad?